In [1]:
library(Seurat)
library(tidyverse)
library(data.table)
library(ggplot2)
library(gsheet)
library(Matrix)
library(data.table)
library(qs)

Loading required package: SeuratObject

Loading required package: sp

‘SeuratObject’ was built with package ‘Matrix’ 1.6.3 but the current
version is 1.6.5; it is recomended that you reinstall ‘SeuratObject’ as
the ABI for ‘Matrix’ may have changed


Attaching package: ‘SeuratObject’


The following object is masked from ‘package:base’:

    intersect


── Attaching core tidyverse packages ────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.4.4     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all

In [2]:
sheet_url <- "REPLACE_WITH_PRIVATE_TRACKING_SHEET_URL"
xenium_tracking_sheet <- gsheet2tbl(sheet_url)

In [3]:
Seurat_object <- qread('/diskmnt/Projects/MetNet_analysis/Colorectal/epeng/Xenium/Xenium_5K/subset_objects/mCRC_Xenium_N18_all_cells.qs')
             

In [4]:
head(Seurat_object)

,orig.ident,nCount_Xenium,nFeature_Xenium,SampleID,CasetteID,group
,<chr>,<dbl>,<int>,<chr>,<list>,<chr>
S16-44514-E7U2__aaaaahpp-1,S16-44514-E7U2,32,25,S16-44514-E7U2,0037123,Smooth muscle
S16-44514-E7U2__aaaafeog-1,S16-44514-E7U2,154,142,S16-44514-E7U2,0037123,Lymphocyte
S16-44514-E7U2__aaaagllj-1,S16-44514-E7U2,292,204,S16-44514-E7U2,0037123,Smooth muscle
S16-44514-E7U2__aaaaifbl-1,S16-44514-E7U2,75,70,S16-44514-E7U2,0037123,Unknown
S16-44514-E7U2__aaabgeka-1,S16-44514-E7U2,107,92,S16-44514-E7U2,0037123,Lymphocyte
S16-44514-E7U2__aaabhilp-1,S16-44514-E7U2,140,129,S16-44514-E7U2,0037123,Lymphocyte
S16-44514-E7U2__aaabjnbg-1,S16-44514-E7U2,166,147,S16-44514-E7U2,0037123,Lymphocyte
S16-44514-E7U2__aaacaakb-1,S16-44514-E7U2,42,41,S16-44514-E7U2,0037123,Unknown
S16-44514-E7U2__aaachgon-1,S16-44514-E7U2,116,103,S16-44514-E7U2,0037123,Lymphocyte


In [5]:
banksy_file <- '/diskmnt/Projects/MetNet_analysis_2/Colorectal/Analysis/Xenium/Banksy/Neighorhood_annotation/mCRC_N16_5K_lambda0.8_VG2000_leiden0.5_Banksy_manual_cell_type_metadata.csv'
Banksy_neighborhoods <- read.table(banksy_file, sep = ',', header = TRUE)

rownames(Banksy_neighborhoods) <- Banksy_neighborhoods$X
Banksy_neighborhoods$group <- NULL
Banksy_neighborhoods$sample_id <- NULL
Banksy_neighborhoods$cell_id <- NULL

common_cells <- intersect(rownames(Banksy_neighborhoods), colnames(Seurat_object))

Banksy_neighborhoods <- Banksy_neighborhoods[common_cells, , drop = FALSE]

Seurat_object <- AddMetaData(object = Seurat_object, metadata = Banksy_neighborhoods)

In [5]:
cell_type_file <- '/diskmnt/Projects/MetNet_analysis_2/Colorectal/Analysis/Xenium/Cell_type/All_cell_type/mCRC_N16_Xenium_banky_celltype_metadata_20250131.csv'
cell_labels <- read.table(cell_type_file, sep = ',', header = TRUE)

rownames(cell_labels) <- cell_labels$X
cell_labels <- cell_labels[, "broad_cell_type", drop = FALSE]

common_cells <- intersect(rownames(cell_labels), colnames(Seurat_object))

cell_labels <- cell_labels[common_cells, , drop = FALSE]

Seurat_object <- AddMetaData(object = Seurat_object, metadata = cell_labels)

In [7]:
unique(Seurat_object$orig.ident)

[1] "S16-44514-E7U2"    "S16-44514-B3U1"    "S15-23849-A2U1"   
 [4] "S15-21169-A1U1"    "S15-1909-D2U1"     "S15-1909-A5U1"    
 [7] "S15-32775-D4U2"    "S16-38794-E2U1"    "S15-32775-A1U2"   
[10] "S16-38794-A4U2"    "HT413C1-Th1K2A1U2" "S13-31378-B1U1"   
[13] "S13-31378-E7U1"    "HT472C1-S1H3FP1U1" "HT472C1-M1FP1U1"  
[16] "S17-16442-AFR1U1"  "S13-18424-B5U1"    "S20-46186-A16U1"

In [7]:
section_ids <- names(Seurat_object@images)

all_spatial_coords <- list()

for (section_id in section_ids) {
  message("Collecting centroids for section: ", section_id)
  spatial_coords <- as.data.frame(Seurat_object@images[[section_id]]$centroids)
  all_spatial_coords[[section_id]] <- spatial_coords
}

Seurat_object@images <- list()


In [8]:
for (section_id in section_ids) {
    print(section_id)
    flush.console()
    
    tmp_section_id <- gsub('\\.','-', section_id)
    out_file <- paste0('seurat_to_python/', tmp_section_id, ".csv")
    
    # Skip if output file already exists
    if (file.exists(out_file)) {
        next
    }
    
    cells_to_keep <- grep(tmp_section_id, rownames(Seurat_object@meta.data), value = TRUE)
    
    if (length(cells_to_keep) == 0) {
      next
    }

    Seurat_object_subset <- subset(Seurat_object, orig.ident == tmp_section_id)

    num_cells1 <- nrow(Seurat_object_subset@meta.data)  # Number of cells in metadata
    num_cells2 <- nrow(all_spatial_coords[[section_id]])  # Number of centroids
    num_cells3 <- dim(GetAssayData(Seurat_object_subset, slot = "data"))[2]  # Number of cells in expression matrix
    if (!all(num_cells1, num_cells2, num_cells3)) {
        stop('different numbers of cells')
    }

    rownames(all_spatial_coords[[section_id]]) <- rownames(Seurat_object_subset@meta.data)

    cell_types <- Seurat_object_subset@meta.data
    
    banksy_cluster <- Seurat_object_subset@meta.data %>% select(banksy_cluster)  # Replace 'group' with the actual column name
    
    expression_data <- as.data.frame(as.matrix(GetAssayData(Seurat_object_subset, slot = "data")))
    
    # Transpose so that cells are rows and genes are columns
    expression_data <- as.data.frame(t(expression_data))

    # Ensure spatial coordinates are in the same order as metadata
    all_spatial_coords[[section_id]] <- all_spatial_coords[[section_id]][rownames(cell_types), ]

    # Merge everything into a final dataframe
    final_df <- cbind(cell_id = rownames(Seurat_object_subset@meta.data), all_spatial_coords[[section_id]], cell_types, banksy_cluster, expression_data)

    fwrite(final_df, file = out_file, row.names = FALSE)
}

[1] "S16.44514.E7U2"
[1] "S16.44514.B3U1"
[1] "S15.23849.A2U1"
[1] "S15.21169.A1U1"
[1] "S15.1909.D2U1"
[1] "S15.1909.A5U1"


Warning message:
“The `slot` argument of `GetAssayData()` is deprecated as of SeuratObject 5.0.0.
ℹ Please use the `layer` argument instead.”
Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 24.3 GiB”


[1] "S15.32775.D4U2"


Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 20.9 GiB”


[1] "S16.38794.E2U1"


Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 2.4 GiB”


... (16 outputs left out)

[1] "S13.18424.B5U1"


Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 11.6 GiB”


[1] "S20.46186.A16U1"


Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 4.7 GiB”


In [ ]:
library(Seurat)
library(dplyr)
library(data.table)  # for fwrite()

# Assume that the cell names in your metadata match the rownames of your centroids.
# For each section (i.e. each element in Seurat_object@images):
for (section in names(Seurat_object@images)) {
  
  # Extract the centroids (spatial coordinates) for this section as a data frame.
  spatial_coords <- as.data.frame(Seurat_object@images[[section]]$centroids)
  
  # Optionally, add a column with the section id.
  spatial_coords$section_id <- section
  
  # Assume that the rownames of the centroids are the cell names.
  # (If not, you might need to adjust how cell names are stored.)
  cells_in_section <- rownames(spatial_coords)
  
  # Subset metadata for cells in this section.
  cell_meta <- Seurat_object@meta.data[cells_in_section, , drop = FALSE]
  cell_types <- cell_meta %>% select(broad_cell_type)
  banksy_cluster <- cell_meta %>% select(banksy_cluster)
  
  # Subset the expression data for these cells.
  # GetAssayData returns a sparse matrix; here we convert only the needed subset to dense.
  expression_data <- as.data.frame(as.matrix(GetAssayData(Seurat_object, slot = "data")[, cells_in_section]))
  # Transpose so that cells are rows and genes are columns.
  expression_data <- as.data.frame(t(expression_data))
  
  # (Optional) Reorder spatial_coords to match the order in the metadata, if needed.
  spatial_coords <- spatial_coords[rownames(cell_meta), ]
  
  # Combine all into one final data frame.
  # Use the cell names from the metadata (or spatial_coords) as the cell_id column.
  final_df <- cbind(cell_id = rownames(cell_meta), spatial_coords, cell_types, banksy_cluster, expression_data)
  
  # Write to CSV; the file name will include the section id.
  fwrite(final_df, file = paste0('seurat_to_python/', section, ".csv"), row.names = FALSE)
  
  cat("Wrote section", section, "with", nrow(final_df), "cells.\n")
}
